# 01 — Bronze Layer Ingestion
**Project**  : Olist Brazilian E-Commerce Data Warehouse  
**Layer**    : Bronze (Raw Ingestion)  
**Author**   : Salman  
**Platform** : Databricks (Unity Catalog + Delta Lake)  

---

## Purpose
This notebook ingests raw CSV files from a Unity Catalog Volume into the Bronze layer as Delta Tables.  
No business transformations are applied at this layer — data is stored **as-is** from the source.  

The only technical handling applied is on the `order_reviews` table:  
the columns `review_comment_title` and `review_comment_message` contain embedded newlines (`\n`, `\r`)  
that cause the CSV parser to misread rows. These are cleaned on-the-fly using `multiLine=True` and `regexp_replace`.

## Data Source
- **Volume Path** : `/Volumes/workspace/default/datasalman/olist_raw/`
- **Format**      : CSV (first row = header)
- **Encoding**    : UTF-8

## Target Tables
| # | Source CSV File | Target Delta Table |
|---|-----------------|--------------------|
| 1 | olist_customers_dataset.csv | workspace.bronze.customers |
| 2 | olist_geolocation_dataset.csv | workspace.bronze.geolocation |
| 3 | olist_order_items_dataset.csv | workspace.bronze.order_items |
| 4 | olist_order_payments_dataset.csv | workspace.bronze.order_payments |
| 5 | olist_order_reviews_dataset.csv | workspace.bronze.order_reviews |
| 6 | olist_orders_dataset.csv | workspace.bronze.orders |
| 7 | olist_products_dataset.csv | workspace.bronze.products |
| 8 | olist_sellers_dataset.csv | workspace.bronze.sellers |
| 9 | product_category_name_translation.csv | workspace.bronze.product_category_name_translation |

## 0. Setup & Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType
)
from datetime import datetime

# Path & catalog configuration
VOLUME_PATH = "/Volumes/workspace/default/datasalman/olist_raw"
CATALOG     = "workspace"
SCHEMA      = "bronze"

# Helper: timestamped log printer
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

log("Setup complete.")
log(f"Volume path : {VOLUME_PATH}")
log(f"Target      : {CATALOG}.{SCHEMA}")

## 1. Create Bronze Schema (if not exists)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
log(f"Schema {CATALOG}.{SCHEMA} is ready.")

## 2. Ingest — customers

In [0]:
start = datetime.now()
log(">> Reading: olist_customers_dataset.csv")

schema_customers = StructType([
    StructField("customer_id",              StringType(),  True),
    StructField("customer_unique_id",       StringType(),  True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city",            StringType(),  True),
    StructField("customer_state",           StringType(),  True),
])

df_customers = spark.read.csv(
    f"{VOLUME_PATH}/olist_customers_dataset.csv",
    header=True,
    schema=schema_customers,
    encoding="UTF-8"
)

log(f"   Row count : {df_customers.count():,}")

(df_customers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.customers"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.customers — {elapsed}s")

## 3. Ingest — geolocation

In [0]:
start = datetime.now()
log(">> Reading: olist_geolocation_dataset.csv")

schema_geolocation = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat",             DoubleType(),  True),
    StructField("geolocation_lng",             DoubleType(),  True),
    StructField("geolocation_city",            StringType(),  True),
    StructField("geolocation_state",           StringType(),  True),
])

df_geolocation = spark.read.csv(
    f"{VOLUME_PATH}/olist_geolocation_dataset.csv",
    header=True,
    schema=schema_geolocation,
    encoding="UTF-8"
)

log(f"   Row count : {df_geolocation.count():,}")

(df_geolocation
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.geolocation"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.geolocation — {elapsed}s")

## 4. Ingest — order_items

In [0]:
start = datetime.now()
log(">> Reading: olist_order_items_dataset.csv")

schema_order_items = StructType([
    StructField("order_id",            StringType(),    True),
    StructField("order_item_id",       IntegerType(),   True),
    StructField("product_id",          StringType(),    True),
    StructField("seller_id",           StringType(),    True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price",               DoubleType(),    True),
    StructField("freight_value",       DoubleType(),    True),
])

df_order_items = spark.read.csv(
    f"{VOLUME_PATH}/olist_order_items_dataset.csv",
    header=True,
    schema=schema_order_items,
    encoding="UTF-8"
)

log(f"   Row count : {df_order_items.count():,}")

(df_order_items
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.order_items"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.order_items — {elapsed}s")

## 5. Ingest — order_payments

In [0]:
start = datetime.now()
log(">> Reading: olist_order_payments_dataset.csv")

schema_order_payments = StructType([
    StructField("order_id",             StringType(),  True),
    StructField("payment_sequential",   IntegerType(), True),
    StructField("payment_type",         StringType(),  True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value",        DoubleType(),  True),
])

df_order_payments = spark.read.csv(
    f"{VOLUME_PATH}/olist_order_payments_dataset.csv",
    header=True,
    schema=schema_order_payments,
    encoding="UTF-8"
)

log(f"   Row count : {df_order_payments.count():,}")

(df_order_payments
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.order_payments"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.order_payments — {elapsed}s")

## 6. Ingest — order_reviews ⚠️ (with embedded newline cleaning)

### Problem
The columns `review_comment_title` and `review_comment_message` contain **embedded newlines**  
(`\n`, `\r`, `\r\n`) that cause the CSV parser to incorrectly split a single review into multiple rows.

### Solution
Read the file using `multiLine=True` so the parser can handle newlines inside quoted fields,  
then strip any remaining newline characters from both text columns using `regexp_replace`.

In [0]:
start = datetime.now()
log(">> Reading: olist_order_reviews_dataset.csv (multiLine mode)")

schema_order_reviews = StructType([
    StructField("review_id",               StringType(),    True),
    StructField("order_id",                StringType(),    True),
    StructField("review_score",            IntegerType(),   True),
    StructField("review_comment_title",    StringType(),    True),
    StructField("review_comment_message",  StringType(),    True),
    StructField("review_creation_date",    TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True),
])

df_order_reviews_raw = spark.read.csv(
    f"{VOLUME_PATH}/olist_order_reviews_dataset.csv",
    header=True,
    schema=schema_order_reviews,
    encoding="UTF-8",
    multiLine=True,
    quote='"',
    escape='"'
)

log(f"   Row count (raw)   : {df_order_reviews_raw.count():,}")

# Strip embedded newlines from text columns
TEXT_COLS = ["review_comment_title", "review_comment_message"]
df_order_reviews = df_order_reviews_raw
for col in TEXT_COLS:
    df_order_reviews = df_order_reviews.withColumn(
        col,
        F.trim(F.regexp_replace(F.col(col), r"[\r\n]+", " "))
    )

log(f"   Row count (clean) : {df_order_reviews.count():,}")

# Validation: ensure no embedded newlines remain
remaining = df_order_reviews.filter(
    F.col("review_comment_title").rlike(r"[\r\n]") |
    F.col("review_comment_message").rlike(r"[\r\n]")
).count()

if remaining == 0:
    log("   [PASS] No embedded newlines remaining.")
else:
    log(f"   [WARN] {remaining} rows still contain newlines — please investigate.")

(df_order_reviews
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.order_reviews"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.order_reviews — {elapsed}s")

## 7. Ingest — orders

In [0]:
start = datetime.now()
log(">> Reading: olist_orders_dataset.csv")

schema_orders = StructType([
    StructField("order_id",                      StringType(),    True),
    StructField("customer_id",                   StringType(),    True),
    StructField("order_status",                  StringType(),    True),
    StructField("order_purchase_timestamp",      TimestampType(), True),
    StructField("order_approved_at",             TimestampType(), True),
    StructField("order_delivered_carrier_date",  TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

df_orders = spark.read.csv(
    f"{VOLUME_PATH}/olist_orders_dataset.csv",
    header=True,
    schema=schema_orders,
    encoding="UTF-8"
)

log(f"   Row count : {df_orders.count():,}")

(df_orders
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.orders"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.orders — {elapsed}s")

## 8. Ingest — products

In [0]:
start = datetime.now()
log(">> Reading: olist_products_dataset.csv")

schema_products = StructType([
    StructField("product_id",                 StringType(),  True),
    StructField("product_category_name",      StringType(),  True),
    StructField("product_name_lenght",        IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty",         IntegerType(), True),
    StructField("product_weight_g",           DoubleType(),  True),
    StructField("product_length_cm",          DoubleType(),  True),
    StructField("product_height_cm",          DoubleType(),  True),
    StructField("product_width_cm",           DoubleType(),  True),
])

df_products = spark.read.csv(
    f"{VOLUME_PATH}/olist_products_dataset.csv",
    header=True,
    schema=schema_products,
    encoding="UTF-8"
)

log(f"   Row count : {df_products.count():,}")

(df_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.products"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.products — {elapsed}s")

## 9. Ingest — sellers

In [0]:
start = datetime.now()
log(">> Reading: olist_sellers_dataset.csv")

schema_sellers = StructType([
    StructField("seller_id",              StringType(),  True),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city",            StringType(),  True),
    StructField("seller_state",           StringType(),  True),
])

df_sellers = spark.read.csv(
    f"{VOLUME_PATH}/olist_sellers_dataset.csv",
    header=True,
    schema=schema_sellers,
    encoding="UTF-8"
)

log(f"   Row count : {df_sellers.count():,}")

(df_sellers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.sellers"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.sellers — {elapsed}s")

## 10. Ingest — product_category_name_translation

In [0]:
start = datetime.now()
log(">> Reading: product_category_name_translation.csv")

schema_translation = StructType([
    StructField("product_category_name",         StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])

df_translation = spark.read.csv(
    f"{VOLUME_PATH}/product_category_name_translation.csv",
    header=True,
    schema=schema_translation,
    encoding="UTF-8"
)

log(f"   Row count : {df_translation.count():,}")

(df_translation
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.product_category_name_translation"))

elapsed = (datetime.now() - start).seconds
log(f">> Done: bronze.product_category_name_translation — {elapsed}s")

## 11. Final Validation — Row Count Check

In [0]:
log("=" * 60)
log("FINAL VALIDATION — BRONZE TABLE ROW COUNTS")
log("=" * 60)

tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_name_translation",
]

expected = {
    "customers":                             99441,
    "geolocation":                         1000163,
    "order_items":                          112650,
    "order_payments":                       103886,
    "order_reviews":                         99224,
    "orders":                                99441,
    "products":                              32951,
    "sellers":                                3095,
    "product_category_name_translation":        71,
}

print(f"\n{'Table':<45} {'Row Count':>12} {'Expected':>12} {'Status':>8}")
print("-" * 80)

all_ok = True
for tbl in tables:
    df    = spark.table(f"{CATALOG}.{SCHEMA}.{tbl}")
    count = df.count()
    exp   = expected[tbl]
    status = "[PASS]" if count == exp else "[FAIL]"
    if count != exp:
        all_ok = False
    print(f"{tbl:<45} {count:>12,} {exp:>12,} {status:>8}")

print("-" * 80)
if all_ok:
    log("[PASS] All tables ingested successfully with matching row counts.")
else:
    log("[FAIL] Some tables have mismatched row counts — please review the output above.")